# laptop_price

## Load and Import

In [2]:
import pandas as pd

train = pd.read_csv('laptop_price_train.csv')
test = pd.read_csv('laptop_price_test.csv')

In [3]:
train.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,Acer,Aspire ES1-523,Notebook,15.6,1366x768,AMD A8-Series 7410 2.2GHz,8GB,1TB HDD,AMD Radeon R5,Windows 10,2.4kg,449.0
1,Lenovo,ThinkPad T460,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,8GB,180GB SSD,Intel HD Graphics 520,Windows 7,1.7kg,1199.0
2,HP,EliteBook 850,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.84kg,1219.0
3,HP,EliteBook 1030,Ultrabook,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core M 6Y75 1.2GHz,16GB,512GB SSD,Intel HD Graphics 515,Windows 10,1.16kg,1965.0
4,Apple,MacBook Air,Ultrabook,11.6,1366x768,Intel Core i5 1.6GHz,4GB,256GB Flash Storage,Intel HD Graphics 6000,Mac OS X,1.08kg,959.0


Vou retirar Product, ja que eh uma coluna pouco informativa

In [4]:
train.drop('Product', axis=1, inplace=True)

## Exploration and planning

O DecisionTree do sklearn nao aceita dados nao numericos. Vamos separar os 
dados para encodar os dados categoricos.

In [5]:
train.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price_euros'],
      dtype='str')

In [6]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1042 non-null   str    
 1   TypeName          1042 non-null   str    
 2   Inches            1042 non-null   float64
 3   ScreenResolution  1042 non-null   str    
 4   Cpu               1042 non-null   str    
 5   Ram               1042 non-null   str    
 6   Memory            1042 non-null   str    
 7   Gpu               1042 non-null   str    
 8   OpSys             1042 non-null   str    
 9   Weight            1042 non-null   str    
 10  Price_euros       1042 non-null   float64
dtypes: float64(2), str(9)
memory usage: 200.6 KB


### Separando colunas em numericas e nao numericas

Verificaremos se todos os pesos estao em 'kg'

In [7]:
(train['Weight'].apply(lambda x: x[-2:]) != 'kg').sum()

np.int64(0)

Como estao, vamos transformar a coluna em numerica

In [8]:
train['Weight'] = train['Weight'].apply(lambda x: x[:-2]).astype('float64')

Faremos o mesmo para RAM:

In [9]:
(train['Ram'].apply(lambda x: x[-2:]) != 'GB').sum()

np.int64(0)

In [10]:
train['Ram'] = train['Ram'].apply(lambda x: x[:-2]).astype('int64')

In [11]:
cat_features = train.select_dtypes('str').columns.tolist()
cat_features

['Company', 'TypeName', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys']

### Investigando a cardinalidade das variaveis numericas

In [12]:
cat_values = train[cat_features].agg(pd.Series.unique)
cat_df = pd.DataFrame(cat_values, columns=['Categories'])
cat_df['n_categories'] = cat_df['Categories'].apply(len)
cat_df.drop('Categories', axis=1, inplace=True)
cat_df

,n_categories
Company,19
TypeName,6
ScreenResolution,38
Cpu,106
Memory,35
Gpu,101
OpSys,9


Produto eh nossa variavel de maior cardinalidade, mas acredito tambem que seria
interessante dropar essa coluna. Vamos testar o gridsearch com e sem ela. Tambem
vamos separar as variaveis categoricas entre variaveis de alta e baixa
cardinalidade.

In [13]:
baixa_card = ['OpSys', 'TypeName']
alta_card = [cat for cat in cat_features if cat not in baixa_card] 
alta_card

['Company', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu']

## GridSearch

### Definindo o pipeline

In [51]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.model_selection import GridSearchCV

card_transformer = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), baixa_card),
    (TargetEncoder(), alta_card),
    remainder='passthrough'
)

card_pipeline = make_pipeline(
    card_transformer,
    DecisionTreeRegressor()
)

tree_params = {
    'decisiontreeregressor__max_depth': [3, 4, 6, 8, 16, None],
    'decisiontreeregressor__min_samples_split': [2, 4, 8, 12],
    'decisiontreeregressor__min_samples_leaf': [2, 4, 8, 12]
}

card_search = GridSearchCV(
    card_pipeline,
    param_grid=tree_params,
    scoring='neg_root_mean_squared_error',
    cv=5
)

X_train = train.drop("Price_euros", axis=1)
y_train = train["Price_euros"]

card_search.fit(X_train, y_train)

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'decisiontreeregressor__max_depth': [3, 4, ...], 'decisiontreeregressor__min_samples_leaf': [2, 4, ...], 'decisiontreeregressor__min_samples_split': [2, 4, ...]}"
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehotencoder', ...), ('targetencoder', ...)]"


In [52]:
card_search.best_params_

{'decisiontreeregressor__max_depth': 8,
 'decisiontreeregressor__min_samples_leaf': 4,
 'decisiontreeregressor__min_samples_split': 12}

In [56]:
test = pd.read_csv('laptop_price_test.csv')
test['Weight'] = test['Weight'].apply(lambda x: x[:-2]).astype('float64')
test['Ram'] = test['Ram'].apply(lambda x: x[:-2]).astype('int64')
modelo = card_search.best_estimator_
result = modelo.predict(test)
pd.Series(result).to_csv('predict_results.csv')